# Leakage

**Notebook 5 of 6.** A model's test score is a promise about the future: this is how well
it will do on penguins nobody has labeled yet. **Leakage** is anything that makes that
promise false, because information about the answer reached the model by a route that
will not exist when the model is actually used.

It comes in two kinds, and this notebook builds one example of each:

| | What leaks | What goes wrong |
| :-- | :-- | :-- |
| **Test leakage** | the test set, into training or into a decision | the **score lies**: it measures data the model has in effect already seen |
| **Target leakage** | the answer, into a feature | the **model is useless**: the feature will not be available when it matters |

Both have the same symptom, a score better than it should be, and both are invisible
unless you look for them.

In [ ]:
# data analysis
import numpy as np
import pandas as pd

# machine learning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [ ]:
df_penguins = pd.read_csv("data/penguins.csv")
labeled = df_penguins.dropna(subset=["sex"]).reset_index(drop=True)

y = labeled["sex"]
measurements = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

## 1. Test leakage: features of pure noise

A thought experiment made real. Give each of the 333 penguins **2,000 columns of random
numbers**. They contain no information about sex whatsoever: they are drawn from a random
number generator, with no connection to the penguins at all.

In [ ]:
rng = np.random.default_rng(0)
noise = pd.DataFrame(
    rng.normal(size=(len(labeled), 2000)),
    columns=[f"noise_{i}" for i in range(2000)],
)
noise.iloc[:5, :6]

With 2,000 columns to choose from, a natural step is to keep only the most promising ones:
the 10 whose values are most strongly correlated with the target. The function below does
that selection.

In [ ]:
def most_correlated(features, target, k=10):
    """Return the names of the `k` columns most correlated with the target.

    Parameters
    ----------
    features : pandas.DataFrame
        Numeric candidate columns.
    target : pandas.Series
        The labels, "MALE" or "FEMALE", in the same row order.
    k : int
        How many columns to keep.

    Returns
    -------
    pandas.Index
        The names of the chosen columns.
    """
    is_male = (target == "MALE").astype(int)
    strength = features.corrwith(is_male).abs()   # correlation, ignoring its sign
    return strength.nlargest(k).index

**The wrong order.** Choose the 10 columns on all 333 penguins, then split, then train and
test.

In [ ]:
chosen = most_correlated(noise, y)

X_train, X_test, y_train, y_test = train_test_split(
    noise[chosen], y, test_size=0.25, random_state=42, stratify=y
)
model = LogisticRegression().fit(X_train, y_train)

print(f"test accuracy, chosen before the split: {model.score(X_test, y_test):.3f}")

**About 76 percent, from random numbers.** That beats the body-mass threshold of notebook
02, which used a real measurement. Nothing about sex is in these columns, so where does it
come from?

From the selection. Among 2,000 random columns, some line up with the 333 labels purely
by chance, including the 84 labels of the future test set. Choosing on all the rows picked
exactly the columns that happen to fit **those particular test penguins**. The test set
helped build the model, and the test score now measures that coincidence.

> **Exercise:**
>
> Do it in the right order: split `noise` and `y` first, with the same settings, then
> choose the 10 columns on the **training set only**, then train and test. What test
> accuracy do the same random numbers reach now?

In [ ]:
# your code: split first, choose on the training set, then train and test

<details><summary>
Click here to compare your answer...
</summary>

**About 48 percent**, no better than a coin, which is the truth about random numbers. The
columns chosen on the training set still fit the training penguins by chance, but that
chance does not carry over to penguins they were not chosen on.

Same data, same model, same selection rule. Only the order changed, and the score moved
by almost 30 points.
</details>

## 2. The rule behind it

The noise columns make the effect large enough to see, but the mistake is general:

> **Anything that learns from data learns from the training set only.**

"Learning" covers more than `fit`. Choosing features, choosing a threshold as in notebook
02, the mean and standard deviation of a `StandardScaler`, the median used to fill missing
values: each is a number computed from data, and each must come from the training set.
The test set is used once, for the final score.

Sometimes the damage is too small to see. Fitting the scaler of notebook 04 on all 333
penguins before splitting gives the same test accuracy as fitting it correctly:

In [ ]:
# wrong: the scaler learns its mean and standard deviation from every penguin
scaled_all = StandardScaler().fit_transform(labeled[measurements])
X_train, X_test, y_train, y_test = train_test_split(
    scaled_all, y, test_size=0.25, random_state=42, stratify=y
)
wrong = LogisticRegression().fit(X_train, y_train).score(X_test, y_test)

# right: split first, then fit the scaler on the training set only
X_train, X_test, y_train, y_test = train_test_split(
    labeled[measurements], y, test_size=0.25, random_state=42, stratify=y
)
scaler = StandardScaler().fit(X_train)
right = LogisticRegression().fit(scaler.transform(X_train), y_train).score(
    scaler.transform(X_test), y_test
)

print(f"scaler fitted before the split: {wrong:.3f}")
print(f"scaler fitted after the split:  {right:.3f}")

Identical, here. Removing 84 penguins barely moves an average of 333, so the leaked
information is tiny. **That is not a reason to do it**: you only know the damage was small
because you checked, and with smaller data, more preprocessing steps or steps that use the
target, it grows. Split first, every time, and the question never comes up.

> **Exercise:**
>
> No code needed. Which of these steps leak the test set into the model?
>
> 1. Filling missing ages with the median age of the whole dataset, then splitting.
> 2. Removing duplicate rows, then splitting.
> 3. Looking at a histogram of the test set to decide whether a feature needs a log
>    transform.
> 4. Choosing a threshold on the training set, then scoring it once on the test set.
> 5. Training 50 different models and reporting the best of their 50 test scores.

Your answers here:

<details><summary>
Click here to compare your answer...
</summary>

1. **Leaks.** The median was computed partly from test rows.
2. **Does not leak, and is the right order.** Removing duplicates learns nothing from the
   data. Doing it *after* the split would be the mistake: a row with a copy in each set
   lets the model be tested on something it has already seen.
3. **Leaks.** A decision was made by looking at the test set, even without any code.
4. **Does not leak.** This is the correct procedure from notebook 02.
5. **Leaks.** Choosing the best of 50 by test score is choosing on the test set, exactly
   like the 2,000 noise columns. The winner's score is optimistic. Comparing models needs
   data of its own, a topic of its own later in the program.
</details>

## 3. Target leakage: a feature that carries the answer

The second kind looks like good feature engineering. Suppose a colleague adds a column: for
each penguin, **the average body mass of penguins like it**, meaning of the same species
and the same sex. It sounds reasonable; notebook 03 showed that mass means different
things for different groups.

In [ ]:
labeled["group_mean_mass"] = (
    labeled.groupby(["species", "sex"])["body_mass_g"].transform("mean")
)
labeled[["species", "sex", "body_mass_g", "group_mean_mass"]].head()

Add it to the four measurements and train, doing everything else correctly.

In [ ]:
features = measurements + ["group_mean_mass"]

X_train, X_test, y_train, y_test = train_test_split(
    labeled[features], y, test_size=0.25, random_state=42, stratify=y
)
scaler = StandardScaler().fit(X_train)
model = LogisticRegression().fit(scaler.transform(X_train), y_train)

print(f"test accuracy: {model.score(scaler.transform(X_test), y_test):.3f}")
pd.Series(model.coef_[0], index=features).round(2)

Test accuracy jumps from 89 to **96 percent**, and the new column gets the largest
coefficient of all. The split was done properly and the scaler fitted on the training set,
so this is not test leakage. The problem is in the column itself.

**The column was computed from `sex`**, the target. Knowing the group average of a penguin
means knowing which sex group it belongs to. The model has not learned to recognize a male
from its measurements; it has partly been told.

The test from notebook 01 exposes it at once: **can the feature be computed at the moment
of prediction?** Try it on the penguins the model exists for, the unlabeled ones:

In [ ]:
unlabeled = df_penguins[df_penguins["sex"].isna()]

group_means = labeled.groupby(["species", "sex"])["body_mass_g"].mean().rename("group_mean_mass")
unlabeled.join(group_means, on=["species", "sex"])[["species", "sex", "group_mean_mass"]]

Empty, all 11 of them. The feature needs the sex to look up the group, so it can never be
filled in for a penguin whose sex is unknown. The 96 percent is a score for a model that
cannot be used.

Note that this leak did not shout. The score rose by seven points, not to 100 percent. Real
leaks are often like that: a pleasant improvement that nobody questions.

> **Exercise:**
>
> The same idea without the leak: compute `species_mean_mass`, the average body mass of
> penguins of the **same species only**. Add it to the four measurements instead of
> `group_mean_mass`, train as above, and compare the test accuracy. Can this feature be
> computed for the unlabeled penguins?

In [ ]:
# your code: a species-only group average, and the model with it

<details><summary>
Click here to compare your answer...
</summary>

About **89 percent**, the same as the four measurements alone, and this one is honest. It
needs only the species, which is known in the field, so it can be filled in for every
unlabeled penguin that has a species. It adds nothing here, because the scaled
measurements already tell the species apart, but it is a legitimate feature.

Same technique, a group average. What decides whether it leaks is only **which columns
the groups are built from**.

A strict reader would add one more point: the species averages were computed on all 333
penguins, test set included. The right way is to compute them on the training set and look
them up for the test set, exactly like the scaler in section 2.
</details>

## Summary: a leakage checklist

**Against test leakage:**

- **Split first**, before any step that computes something from the data.
- Fit every step, from feature choice to scaler to fill value, on the **training set
  only**, then apply it to the test set.
- Use the test set **once**. Every extra look turns it a little into training data.

**Against target leakage**, for every feature:

- Is it **known at the moment of prediction**?
- Was it **computed without the target**?
- Does it exist **because of the outcome**, recorded only after the event?

**And one habit for both:** a score that looks too good, or a single feature that dominates
the model, is a reason to go looking for a leak before celebrating.

Next, and optional, in `06_titanic_exercise.ipynb`: a real dataset, with a real leak in it that nobody
planted. Your job is to find it before it fools you.